# Lab 6 — Observability with OpenTelemetry

Enable Strands console tracing and run a tool-enabled incident Agent; inspect model/tool spans. Avoid sending prompts with secrets to trace destinations.

**Run cells from top to bottom.** Execute `00_Setup.ipynb` first. Each notebook is intended to be opened from the `notebooks/` folder.

In [1]:
import os, sys, logging
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from strands.telemetry import StrandsTelemetry
from common import build_agent, get_incident_metrics

## 1. Enable telemetry in the kernel

Initialize the console exporter **once** per kernel. If you repeat this cell, you may create duplicate instrumentation depending on SDK/runtime versions. Restart the kernel when needed.

In [2]:
logging.basicConfig(level=logging.INFO)
StrandsTelemetry().setup_console_exporter()
print("OpenTelemetry console exporter configured; traces may contain prompts and tool inputs.")

INFO:strands.telemetry.config:Initializing tracer
INFO:strands.telemetry.config:Enabling console export


OpenTelemetry console exporter configured; traces may contain prompts and tool inputs.


## 2. Invoke a tool-enabled Agent

In [3]:
traced_agent = build_agent(
    name="traced_monitor",
    description="Inspects synthetic operational metrics.",
    system_prompt="Call get_incident_metrics for INC-1042 and return evidence with numbers. No production actions.",
    tools=[get_incident_metrics],
)
traced_result = traced_agent("Inspect INC-1042 metrics using the tool and summarize the key anomalies.")
print("Agent result:", traced_result)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:strands.telemetry.metrics:Creating Strands MetricsClient


Sure! Let me fetch the synthetic lab metrics for INC-1042 right away.
Tool #1: get_incident_metrics
{
    "name": "chat",
    "context": {
        "trace_id": "0xc263643468a96332741ade5999e819b2",
        "span_id": "0x5e9813d4b796373f",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb29c39a3a47a8943",
    "start_time": "2026-11-11T08:18:52.467483Z",
    "end_time": "2026-11-11T08:18:55.309481Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-11-11T08:18:52.467487+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "gen_ai.request.model": "global.anthropic.claude-sonnet-4-6",
        "gen_ai.event.end_time": "2026-11-11T08:18:55.309382+00:00",
        "gen_ai.usage.prompt_tokens": 1042,
        "gen_ai.usage.input_tokens": 1042,
        "gen_ai.usage.completion_tokens": 81,
        "gen_ai.usage.output_tokens": 81,
        "gen_ai.usa

## 3. Inspect traces and logs

In Jupyter output, find model and tool spans and note duration, available token fields, and errors. Not all span attributes are emitted in every model/provider/version. Mask sensitive prompts before exporting traces.

In [ ]:
print("Observation sheet")
print("Model invocation span visible? YES / NO")
print("Tool invocation span visible? YES / NO")
print("Token usage visible? YES / NO / NOT EXPOSED")
print("Any errors? __________________________________")

**Checkpoint:** Verify visible telemetry. Compare end-to-end time and model/tool calls with the Swarm and Workflow labs, using only data your traces actually expose.